# Query Hive Tables via PyHive

Connects directly to **HiveServer2** (`hive-server2:10000`) using the `pyhive` library.

> **Why not `enableHiveSupport()`?**  
> The Spark binary installed (`spark-3.5.1-bin-hadoop3`) does not include Hive assembly JARs, so `enableHiveSupport()` fails with `'JavaPackage' object is not callable`.  
> PyHive is a pure-Python Hive client that works without any extra JARs.


## 1. Install Required Libraries


In [1]:
import subprocess, sys

# sasl/thrift-sasl require libsasl2-dev (not available in this container).
# Since HiveServer2 uses auth="NONE", plain thrift transport is sufficient.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "pyhive", "thrift", "pandas"
])
print("Libraries installed.")


Libraries installed.


## 2. Connect to HiveServer2


In [2]:
from pyhive import hive
import pandas as pd

try:
    conn = hive.connect(
        host="hive-server2",
        port=10000,
        database="default",
        auth="NONE",
    )
    cursor = conn.cursor()
    print("Connected to HiveServer2 successfully!")
except Exception as e:
    print(f"Connection failed: {e}")
    print("Make sure 'hive-server2' container is running and reachable on port 10000.")


Connected to HiveServer2 successfully!


## 3. List Databases and Tables


In [3]:
cursor.execute("SHOW DATABASES")
print("=== Databases ===")
for row in cursor.fetchall():
    print(" -", row[0])

cursor.execute("SHOW TABLES")
print("\n=== Tables in 'default' ===")
for row in cursor.fetchall():
    print(" -", row[0])  # HiveServer2 returns single-element tuples for SHOW TABLES


=== Databases ===
 - default

=== Tables in 'default' ===
 - customers
 - locations
 - order_items
 - orders
 - products


In [5]:
# Strip "table." prefix from column names (HiveServer2 returns "customers.customer_id" etc.)
def fetch_df(cur, sql):
    cur.execute(sql)
    cols = [d[0].split(".")[-1] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

df_customers = fetch_df(cursor, "SELECT * FROM customers")
print(f"Total rows: {len(df_customers)}")
display(df_customers.head(100))


Total rows: 793


,customer_id,customer_name,segment
0,CG-12520,Claire Gute,Consumer
1,DV-13045,Darrin Van Huff,Corporate
2,SO-20335,Sean O'Donnell,Consumer
3,BH-11710,Brosina Hoffman,Consumer
4,AA-10480,Andrew Allen,Consumer
...,...,...,...
95,LC-17140,Logan Currie,Consumer
96,HK-14890,Heather Kirkland,Corporate
97,LE-16810,Laurel Elliston,Consumer
98,JH-15985,Joseph Holt,Consumer


## 5. Explore Customer Data


In [6]:
print("=== Shape ===")
print(df_customers.shape)

print("\n=== Column info ===")
df_customers.info()

print("\n=== Segment distribution ===")
display(df_customers["segment"].value_counts().reset_index().rename(columns={"segment": "count", "index": "segment"}))


=== Shape ===
(793, 3)

=== Column info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 793 entries, 0 to 792
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customer_id    793 non-null    object
 1   customer_name  793 non-null    object
 2   segment        793 non-null    object
dtypes: object(3)
memory usage: 18.7+ KB

=== Segment distribution ===


,count,count
0,Consumer,409
1,Corporate,236
2,Home Office,148


## 6. Filter Customers by Segment


In [7]:
segment_filter = "Consumer"  # change to "Corporate" or "Home Office"

df_segment = fetch_df(
    cursor,
    f"SELECT customer_id, customer_name, segment FROM customers WHERE segment = '{segment_filter}' ORDER BY customer_name"
)
print(f"Customers in segment '{segment_filter}': {len(df_segment)}")
display(df_segment)


Customers in segment 'Consumer': 409


,customer_id,customer_name,segment
0,AB-10015,Aaron Bergman,Consumer
1,AS-10090,Adam Shillingsburg,Consumer
2,AB-10105,Adrian Barton,Consumer
3,AB-10150,Aimee Bixby,Consumer
4,AB-10165,Alan Barnes,Consumer
...,...,...,...
404,VM-21835,Vivian Mathis,Consumer
405,WB-21850,William Brown,Consumer
406,XP-21865,Xylona Preis,Consumer
407,ZC-21910,Zuschuss Carroll,Consumer


## 7. Query `products` Table


In [8]:
df_products = fetch_df(cursor, "SELECT * FROM products")
print(f"Total rows: {len(df_products)}")
display(df_products.head(20))


Total rows: 1862


,product_id,product_name,category,sub_category
0,FUR-BO-10001798,Bush Somerset Collection Bookcase,Furniture,Bookcases
1,FUR-CH-10000454,"""Hon Deluxe Fabric Upholstered Stacking Chairs","Rounded Back""",Furniture
2,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,Office Supplies,Labels
3,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangular Table,Furniture,Tables
4,OFF-ST-10000760,Eldon Fold 'N Roll Cart System,Office Supplies,Storage
5,FUR-FU-10001487,"""Eldon Expressions Wood and Plastic Desk Acces...","Cherry Wood""",Furniture
6,OFF-AR-10002833,Newell 322,Office Supplies,Art
7,TEC-PH-10002275,Mitel 5320 IP Phone VoIP phone,Technology,Phones
8,OFF-BI-10003910,DXL Angle-View Binders with Locking Rings by S...,Office Supplies,Binders
9,OFF-AP-10002892,Belkin F5C206VTEL 6 Outlet Surge,Office Supplies,Appliances


## 8. Filter Products by Category


In [9]:
category_filter = "Furniture"  # change to "Technology" or "Office Supplies"

df_cat = fetch_df(
    cursor,
    f"SELECT * FROM products WHERE category = '{category_filter}' ORDER BY product_name"
)
print(f"Products in category '{category_filter}': {len(df_cat)}")
display(df_cat)


Products in category 'Furniture': 240


,product_id,product_name,category,sub_category
0,FUR-TA-10004152,"""Barricks 18"""" x 48"""" Non-Folding Utility Tabl...",Furniture,Tables
1,FUR-TA-10002903,"""Bevis Round Bullnose 29"""" High Table Top""",Furniture,Tables
2,FUR-TA-10001932,"""Chromcraft 48"""" x 96"""" Racetrack Double Pedes...",Furniture,Tables
3,FUR-TA-10003238,"""Chromcraft Bull-Nose Wood 48"""" x 96"""" Rectang...",Furniture,Tables
4,FUR-FU-10004904,"""Eldon """"L"""" Workstation Diamond Chairmat""",Furniture,Furnishings
...,...,...,...,...
235,FUR-FU-10001057,Tensor Track Tree Floor Lamp,Furniture,Furnishings
236,FUR-FU-10002874,Ultra Commercial Grade Dual Valve Door Closer,Furniture,Furnishings
237,FUR-FU-10001889,Ultra Door Pull Handle,Furniture,Furnishings
238,FUR-FU-10002268,Ultra Door Push Plate,Furniture,Furnishings


In [ ]:
## 9. Close Connection


In [10]:
cursor.close()
conn.close()
print("Connection closed.")


Connection closed.
